<a href="https://colab.research.google.com/github/JosephNathaniel/ARENA/blob/main/chapter1_transformer_interp/exercises/part2_intro_to_mech_interp/induction-heads-rigorously.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Imports

In [4]:
import torch as t
import torch.nn as nn
import torch.nn.functional as F
import math
import einops

# Part 1 -- a minimal, analytic induction head

In [ ]:
class AttentionBlock(nn.Module):
    """
    A single attention block according to the specified constraints:
    - 1 head
    - d_head = d_model
    - No MLP block
    - No LayerNorm
    - Positional embeddings added to Q and K inputs, not the residual stream.
    """
    def __init__(self, d_model: int, max_seq_len: int):
        super().__init__()
        self.d_model = d_model
        # Since num_heads = 1 and d_head = d_model, the projection dimensions are straightforward
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
        # Scale factor for scaled dot-product attention
        self.scale = d_model ** -0.5

        # Create maximally sized causal mask. False is masked, True is unmasked
        self.maximal_causal_mask = t.tril(t.ones(max_seq_len, max_seq_len), diagonal=-1).bool()

    def forward(self, x: t.Tensor, pos_emb: t.Tensor, mask: t.Tensor | None = None) -> t.Tensor:
        """
        Forward pass for the Attention Block.

        Args:
            x: Input tensor (residual stream). Shape: (batch_size, seq_len, d_model)
            pos_emb: Positional embeddings. Shape: (1, seq_len, d_model) or (batch_size, seq_len, d_model)
            mask: Optional attention mask. Shape: (batch_size, seq_len, seq_len) or similar broadcastable.
                  Positions with `False` or `0` will be masked out.

        Returns:
            Output tensor after attention and output projection. Shape: (batch_size, seq_len, d_model)
        """
        batch_size, seq_len, _ = x.shape

        # 1. Add positional embeddings *before* Q and K projections
        #    Value stream does not get positional embeddings added here.
        q_input = x + pos_emb
        k_input = x + pos_emb
        v_input = x # V uses the original residual stream state

        # 2. Project inputs to Q, K, V
        Q = self.W_q(q_input) # Shape: (batch_size, seq_len, d_model)
        K = self.W_k(k_input) # Shape: (batch_size, seq_len, d_model)
        V = self.W_v(v_input) # Shape: (batch_size, seq_len, d_model)

        # 3. Calculate scaled dot-product attention scores
        # (batch, seq_len, d_model) @ (batch, d_model, seq_len) -> (batch, seq_len, seq_len)
        attn_scores = einops.einsum(Q, K, "b sq d, b sk d -> b sq sk") * self.scale

        # 4. Apply attention mask (if provided)
        if mask is not None:
            # Ensure mask has compatible dimensions for broadcasting
            if mask.dim() == 2:
                 mask = mask[:seq_len, :seq_len] # this will deal with e.g. overly large maximal causal mask
                 mask = mask.unsqueeze(0) # (seq_len, seq_len) -> (1, seq_len, seq_len)
            elif mask.dim() == 3: # (batch, 1, seq_len) or (batch, seq_len, 1) -> (batch, seq_len, seq_len)
                 raise NotImplementedError

            # Masked positions are filled with a large negative value
            attn_scores = attn_scores.masked_fill(mask == False, -1e9) # Use a large negative number

        # 5. Apply softmax to get attention probabilities
        attn_probs = F.softmax(attn_scores, dim=-1) # Shape: (batch_size, seq_len, seq_len)

        # 6. Calculate weighted sum of Values
        # (batch, seq_len, seq_len) @ (batch, seq_len, d_model) -> (batch, seq_len, d_model)
        attn_output = einops.einsum(attn_probs, V, "b sq sk, b sk d -> b sq d")

        # 7. Apply output projection
        output = self.W_o(attn_output) # Shape: (batch_size, seq_len, d_model)

        return output


class SimpleTransformer(nn.Module):
    """
    A simple 2-layer, attention-only Transformer with specific positional embedding handling.
    """
    def __init__(self, vocab_size: int, d_model: int, num_layers: int, max_seq_len: int):
        super().__init__()
        assert num_layers == 2, "This specific implementation is hardcoded for num_layers=2"
        self.d_model = d_model
        self.max_seq_len = max_seq_len

        # Token Embedding layer
        self.token_embedding = nn.Embedding(vocab_size, d_model)

        # Positional Embedding layer (will be added before Q and K, not to residual stream directly)
        self.positional_embedding = nn.Embedding(max_seq_len, d_model)

        # Stack of Attention Blocks
        self.layers = nn.ModuleList([AttentionBlock(d_model, max_seq_len) for _ in range(num_layers)])

        # Final linear layer to map back to vocabulary size (optional, for tasks like LM)
        self.unembed = nn.Linear(d_model, vocab_size, bias=False)

        # Optional: Tie weights between embedding and unembedding layers
        # self.token_embedding.weight = self.unembed.weight # Often done in language models


    def forward(self, input_ids: t.Tensor, attention_mask: t.Tensor | None = None) -> t.Tensor:
        """
        Forward pass for the Simple Transformer.

        Args:
            input_ids: Input token IDs. Shape: (batch_size, seq_len)
            attention_mask: Optional mask to prevent attention to certain positions (e.g., padding).
                            Shape: (batch_size, seq_len). Positions with 1 are kept, 0 are masked.
                            Will be converted to the appropriate shape for attention scores.

        Returns:
            Logits tensor. Shape: (batch_size, seq_len, vocab_size)
        """
        batch_size, seq_len = input_ids.shape
        if seq_len > self.max_seq_len:
             raise ValueError(f"Input sequence length ({seq_len}) exceeds model's max sequence length ({self.max_seq_len})")


        # 1. Get Token Embeddings
        token_emb = self.token_embedding(input_ids) # Shape: (batch_size, seq_len, d_model)

        # 2. Get Positional Embeddings
        # Create position IDs (0, 1, 2, ..., seq_len-1)
        position_ids = t.arange(0, seq_len, device=input_ids.device).unsqueeze(0) # Shape: (1, seq_len)
        pos_emb = self.positional_embedding(position_ids) # Shape: (1, seq_len, d_model)
        # pos_emb will broadcast correctly to (batch_size, seq_len, d_model) when added

        # 3. Prepare Attention Mask for multi-head attention format (even though we have 1 head)
        attn_mask_processed = None
        if attention_mask is not None:
            # Standard padding masks are (batch, seq_len). Need (batch, seq_len, seq_len)
            # where mask[b, i, j] = 0 indicates attention from token i to token j is masked for batch b.
            # We want mask=0 where attention_mask=0
            attn_mask_processed = attention_mask.unsqueeze(1) * attention_mask.unsqueeze(2)
            # Shape: (batch_size, seq_len, seq_len)

        # 4. Pass through layers with residual connections
        # Initial residual stream is just the token embeddings
        residual_stream = token_emb

        for layer in self.layers:
            # Pass the current residual stream and the *same* positional embeddings each time
            attn_output = layer(residual_stream, pos_emb, mask=attn_mask_processed)
            # Add the output of the attention block back to the residual stream
            residual_stream = residual_stream + attn_output

        # 5. Final unembedding layer (optional)
        logits = self.unembed(residual_stream) # Shape: (batch_size, seq_len, vocab_size)

        return logits

In [ ]:
# --- Example Usage ---
if __name__ == '__main__':
    # Configuration
    VOCAB_SIZE = 1000
    D_MODEL = 128 # d_head will also be 128
    NUM_LAYERS = 2 # As requested
    MAX_SEQ_LEN = 64
    BATCH_SIZE = 4
    SEQ_LEN = 30 # Example sequence length (<= MAX_SEQ_LEN)

    # Create dummy input data
    dummy_input_ids = t.randint(0, VOCAB_SIZE, (BATCH_SIZE, SEQ_LEN))
    # Create a dummy attention mask (e.g., mask out last 5 tokens for padding)
    dummy_attention_mask = t.ones(BATCH_SIZE, SEQ_LEN)
    if SEQ_LEN > 5:
       dummy_attention_mask[:, -5:] = 0

    # Instantiate the model
    model = SimpleTransformer(
        vocab_size=VOCAB_SIZE,
        d_model=D_MODEL,
        num_layers=NUM_LAYERS,
        max_seq_len=MAX_SEQ_LEN
    )

    print(f"Model:\n{model}")
    num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\nTotal trainable parameters: {num_params:,}")

    # Perform a forward pass
    try:
        with t.no_grad(): # Disable gradient calculations for inference
            output_logits = model(dummy_input_ids, attention_mask=dummy_attention_mask)

        print(f"\nInput shape: {dummy_input_ids.shape}")
        print(f"Attention mask shape: {dummy_attention_mask.shape}")
        print(f"Output logits shape: {output_logits.shape}")
        print("\nForward pass successful!")

        # Verify masking (optional check)
        # Create a fully masked input and check if output changes significantly for masked parts
        masked_input = t.randint(0, VOCAB_SIZE, (1, SEQ_LEN))
        full_mask = t.zeros(1, SEQ_LEN)
        out_fully_masked = model(masked_input, attention_mask=full_mask)
        out_no_mask = model(masked_input, attention_mask=t.ones(1,SEQ_LEN))
        # This is a basic check; rigorous checks would involve inspecting attention scores.
        print(f"Output norm with no mask: {t.norm(out_no_mask)}")
        print(f"Output norm with full mask: {t.norm(out_fully_masked)} (should be similar if initial embedding dominates)")

    except Exception as e:
        print(f"\nAn error occurred during forward pass: {e}")
        import traceback
        traceback.print_exc()


    # Example with sequence length exceeding max_seq_len
    print("\nTesting sequence length > max_seq_len:")
    long_input_ids = t.randint(0, VOCAB_SIZE, (BATCH_SIZE, MAX_SEQ_LEN + 1))
    try:
         with t.no_grad():
             output_logits = model(long_input_ids)
    except ValueError as e:
         print(f"Caught expected error: {e}")